In [ ]:
# --- cell 1 · bootstrap（任务书 §0.1 末、§0.2）—— 五个 B notebook 里这一格完全相同 ---
import pathlib, sys

# (1) 向上搜索含 npnn/ 的目录 —— 不依赖 Jupyter 的启动目录
def _find_taskb_root(start):
    for p in (start, *start.parents):
        if (p / 'npnn').is_dir():
            return p
        if (p / 'task_b' / 'npnn').is_dir():      # 从项目根启动 Jupyter 的情况
            return p / 'task_b'
    return None

_here = pathlib.Path.cwd().resolve()
TASKB_ROOT = _find_taskb_root(_here)
assert TASKB_ROOT is not None, f'从 {_here} 逐级往上找不到含 npnn/ 的目录；请从 task_b/ 或项目根启动'
sys.path.insert(0, str(TASKB_ROOT))

# (2) 物理挡住「误 import A 的 src」：把项目根从 sys.path 里摘掉（任务书 §0.1.2）
_root = TASKB_ROOT.parent
sys.path[:] = [e for e in sys.path if pathlib.Path(e or '.').resolve() != _root]

# (3) 线程预算：必须在 import numpy 之前（BLAS 首次 import 时锁死线程数）
import config
config.limit_threads()

# (4) 环境留痕 —— 报告 B.1 要用：哪个解释器、哪个 numpy
import numpy as np
print('sys.executable:', sys.executable)
print('sys.prefix    :', sys.prefix)
print('numpy         :', np.__version__)
print('TASKB_ROOT    :', TASKB_ROOT)


# 13 · 最终分析与图表（第 4 阶段素材）

对应任务书：§4.2（图表清单）、§4.4（Run All 要求）。

**只读 `runs/` 里落盘的数据画图，绝不重新训练** —— 这样本 notebook 从头 Execute All 只要几秒。
（§4.4 专门警告过：B 的 notebook 互相依赖且训练很慢，所以图表一律从落盘文件重画。）

✍️ 产出（对着 §4.2 的 9 项清单打勾）：
- [ ] 数值梯度的 h 扫描曲线（对数轴）
- [ ] 模块验证汇总表（每模块 × 前向误差 / 反向误差 / 数值梯度误差）
- [ ] Conv2d 多形状对拍表（覆盖 stride / kernel / 不整除等边界）
- [ ] im2col vs 朴素循环耗时柱状图（+ numpy vs torch 速度比）
- [ ] 整网 loss 对拍图（L2 核心图：两条线 + 差值子图）
- [ ] numpy vs torch 精度曲线（两条 test acc）
- [ ] ★ SN 权重演化曲线（w_BN / w_IN / w_LN 随训练变化）
- [ ] SN 退化测试表（三行，各报 max|Δ|）
- [ ] 环境与版本留痕 + 纯度守卫测试的输出

图输出到 `report/figs/`（供 main.tex 引用）；原始数据留在 `runs/`。

⚠️ 所有误差都给量级（`max|Δ| = 3.2e-11` 这种），不要写「基本一致」。
「报告里的每个数字都能追溯到某次 run 或某个测试脚本」是提交清单的一条。
